# MAXMIZE MUSIC AI: EDM Classifier Training

Run in Google Colab with a GPU runtime. This notebook trains a first research baseline only. It does not automatically download music. Add only tracks with individually verified ML-training and commercial-use rights.

Expected Drive layout:
- `MAXMIZE_MUSIC/dataset/manifest.csv`
- Audio files at the exact `local_path` values in the manifest (absolute paths under `MAXMIZE_MUSIC/dataset/` are easiest).

Do not upload NCS, NC-licensed, research-only, ripped, or unclear-rights audio. Keep the original license and rights evidence for every track.


In [ ]:
!pip -q install librosa soundfile scikit-learn pandas matplotlib


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
ROOT = Path('/content/drive/MyDrive/MAXMIZE_MUSIC')
DATA_DIR = ROOT / 'dataset'
MANIFEST = DATA_DIR / 'manifest.csv'
OUTPUT_DIR = ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Manifest:', MANIFEST)
print('Manifest exists:', MANIFEST.exists())


## 1. Audit rights and manifest

The audit fails closed. A human must verify the original track page, license, and rights evidence before marking permission as `yes`. Do not mark permissions based on assumptions.


In [ ]:
import csv, hashlib, re, json, os
import pandas as pd
assert MANIFEST.exists(), 'Upload manifest.csv to MAXMIZE_MUSIC/dataset first.'
df = pd.read_csv(MANIFEST).fillna('')
required = {'track_id','sha256','local_path','source_provider','source_url','license_id','license_url','ml_training_permission','commercial_use_permission','rights_evidence_url','subgenre','split'}
missing = required - set(df.columns)
assert not missing, f'Missing manifest columns: {sorted(missing)}'
allowed = {'CC0-1.0','CC-BY-4.0','CC-BY-3.0','PUBLIC-DOMAIN'}
assert len(df) > 0, 'Manifest has no rows.'
assert df.track_id.is_unique, 'Duplicate track_id values.'
assert df.sha256.str.lower().is_unique, 'Duplicate audio hashes.'
assert set(df.license_id.str.upper()).issubset(allowed), 'Unapproved license found.'
assert df.ml_training_permission.astype(str).str.lower().isin(['yes','true','1','verified']).all(), 'ML permission not verified for every row.'
assert df.commercial_use_permission.astype(str).str.lower().isin(['yes','true','1','verified']).all(), 'Commercial permission not verified for every row.'
assert df.split.str.lower().isin(['train','validation','test','holdout']).all(), 'Invalid split name.'
for col in ['source_url','license_url','rights_evidence_url','source_provider','local_path','subgenre']:
    assert df[col].astype(str).str.strip().ne('').all(), f'Missing {col} value.'
for row in df.itertuples():
    p = Path(row.local_path)
    assert p.is_file(), f'Audio file missing: {p}'
    digest = hashlib.sha256(p.read_bytes()).hexdigest()
    assert digest.lower() == str(row.sha256).lower(), f'SHA-256 mismatch: {row.track_id}'
print('PASS: structural, permission-field, file and hash checks. Human rights review is still mandatory.')
display(df.groupby(['subgenre','split']).size().unstack(fill_value=0))


## 2. Prepare audio features

The first baseline uses log-mel spectrograms and a compact CNN. Every recording stays in its preassigned artist/track-separated split. Do not put excerpts from the same original recording in different splits.


In [ ]:
import numpy as np, librosa, tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter
SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)
SR = 22050
DURATION = 15
N_MELS = 128
MAX_FRAMES = 646
def audio_to_mel(path):
    y, _ = librosa.load(path, sr=SR, mono=True, duration=DURATION)
    if len(y) < SR * 5:
        raise ValueError(f'Audio shorter than 5 seconds: {path}')
    mel = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=2048, hop_length=512, n_mels=N_MELS, fmin=30, fmax=11000)
    logmel = librosa.power_to_db(mel, ref=np.max)
    logmel = np.clip((logmel + 80.0) / 80.0, 0.0, 1.0)
    if logmel.shape[1] < MAX_FRAMES:
        logmel = np.pad(logmel, ((0,0),(0,MAX_FRAMES-logmel.shape[1])))
    else:
        logmel = logmel[:, :MAX_FRAMES]
    return logmel.astype('float32')[..., None]
features, valid_rows = [], []
for row in df.itertuples():
    try:
        features.append(audio_to_mel(row.local_path)); valid_rows.append(row)
    except Exception as e:
        print('SKIP', row.track_id, str(e))
assert features, 'No usable audio files.'
X = np.stack(features)
meta = pd.DataFrame([r._asdict() for r in valid_rows])
meta['split'] = meta['split'].str.lower()
meta['subgenre'] = meta['subgenre'].astype(str).str.strip()
le = LabelEncoder(); y = le.fit_transform(meta.subgenre)
train_idx = np.where(meta.split.values == 'train')[0]
val_idx = np.where(meta.split.values == 'validation')[0]
test_idx = np.where(meta.split.values.isin(['test','holdout']))[0]
assert len(train_idx) and len(val_idx) and len(test_idx), 'Need train, validation, and test/holdout examples.'
assert len(set(y[train_idx])) >= 2, 'Need at least two subgenres in training split.'
assert set(y[val_idx]).issubset(set(y[train_idx])), 'Validation contains a class absent from training.'
assert set(y[test_idx]).issubset(set(y[train_idx])), 'Test contains a class absent from training.'
print('Classes:', list(le.classes_)); print('Split counts:', len(train_idx), len(val_idx), len(test_idx))


## 3. Train baseline classifier

This baseline is not production-ready. Small or imbalanced datasets will overfit. Add more verified examples per subgenre and inspect the held-out metrics before deployment.


In [ ]:
num_classes = len(le.classes_)
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(N_MELS, MAX_FRAMES, 1)),
    tf.keras.layers.Conv2D(16, 3, padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D((2,4)),
    tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D((2,4)),
    tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
weights = compute_class_weight(class_weight='balanced', classes=np.unique(y[train_idx]), y=y[train_idx])
class_weights = {int(c): float(w) for c,w in zip(np.unique(y[train_idx]), weights)}
callbacks = [tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True), tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=3, factor=0.5)]
history = model.fit(X[train_idx], y[train_idx], validation_data=(X[val_idx], y[val_idx]), epochs=40, batch_size=16, class_weight=class_weights, callbacks=callbacks)


## 4. Evaluate on untouched test data

Review per-class precision, recall, F1 and confusion matrix. Do not report training accuracy as model quality. If a subgenre has too few test examples, collect more data before making claims about it.


In [ ]:
probs = model.predict(X[test_idx], verbose=0)
pred = probs.argmax(axis=1)
print(classification_report(y[test_idx], pred, labels=np.arange(num_classes), target_names=le.classes_, zero_division=0))
print('Confusion matrix rows=true, columns=predicted:')
print(confusion_matrix(y[test_idx], pred, labels=np.arange(num_classes)))
print('Test accuracy:', float((pred == y[test_idx]).mean()))


## 5. Export model and label map

The exported TensorFlow Lite model expects a normalized log-mel tensor shaped `[1, 128, 646, 1]`. Android preprocessing must match the notebook exactly. Do not connect the model to the production app until inference parity and device tests pass.


In [ ]:
saved_model_dir = OUTPUT_DIR / 'edm_classifier_savedmodel'
model.export(str(saved_model_dir))
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
(OUTPUT_DIR / 'maxmize_edm_classifier.tflite').write_bytes(tflite_model)
(OUTPUT_DIR / 'labels.json').write_text(json.dumps({'labels': le.classes_.tolist(), 'sample_rate': SR, 'duration_seconds': DURATION, 'n_mels': N_MELS, 'max_frames': MAX_FRAMES, 'normalization': 'clip((logmel_db + 80) / 80, 0, 1)', 'model_version': '0.1.0-baseline'}, indent=2))
print('Exported:', OUTPUT_DIR)
print('Copy the .tflite and labels.json into a versioned model release only after review.')
